# Sector Seasonal Dip-Buy · Backtest

**The rule.** At the end of every quarter, look at each S&P 500 sector ETF. Buy it for the coming quarter when
both of these are true:

1. Its **average return in the coming calendar quarter**, over the past *N* years, is **positive**.
2. Its **return in the quarter that just ended is below that average**.

*Example.* At the end of Q3, Consumer Staples returned −5% for Q3. Its average Q4 return over the past 5 years is
+3%. Both conditions hold, so the rule buys XLP for Q4. The same test runs on every sector each quarter. All
qualifying sectors are held in equal weight until the next quarter end. If none qualifies, the money sits in
Treasury bills.

**What this notebook answers**

| Section | Question |
|---|---|
| 1 | What did the rule do in the most recent quarter? |
| 2 | How did it perform against the S&P 500, net of trading costs? |
| 3 | Which sectors did it buy, and how did those trades go? |
| 4 | Does the dip filter help, or would buying every seasonal favourite have done better? |
| 5 | Which lookback (3, 5, 7 or 10 years) and threshold work best? |
| 6 | Does the best setting still work on years it was not chosen on? |
| 7 | Could random sector picks have done as well? |
| 8 | What does the rule say for the upcoming quarter? |
| 9 | Scorecard |

**How to run.** Install the packages once with `pip install yfinance pandas numpy matplotlib`, then choose
**Run → Run All Cells**. It downloads about 27 years of daily prices from Yahoo Finance, which needs an internet
connection and takes about a minute. Nothing is saved to disk. Every setting is in the next cell.

In [ ]:
# ---------------------------------------------------------------- settings
SECTORS = {                     # sector name -> SPDR sector ETF
    "Communication Services": "XLC",
    "Consumer Discretionary": "XLY",
    "Consumer Staples": "XLP",
    "Energy": "XLE",
    "Financials": "XLF",
    "Healthcare": "XLV",
    "Industrials": "XLI",
    "Information Technology": "XLK",
    "Materials": "XLB",
    "Real Estate": "XLRE",
    "Utilities": "XLU",
}
BENCHMARK = "SPY"               # S&P 500 ETF, the benchmark
CASH_TICKER = "^IRX"            # 13-week T-bill yield, used for idle cash and the Sharpe ratio
START = "1998-12-01"            # the sector ETFs start trading in December 1998

LOOKBACK_YEARS = 5              # N: years averaged for the coming quarter (sections 1-4, 7-8)
MIN_SEASONAL_AVG = 0.0          # condition 1: the average must be above this (0 = positive)
DIP_MARGIN = 0.0                # condition 2: the last quarter must be below the average by more than this
EMPTY_QUARTER = "cash"          # when nothing qualifies: "cash" (T-bills), "spy" or "equal_weight"
EXECUTION_LAG_DAYS = 1          # trade at the close this many trading days after the quarter ends
COST_BPS = 5                    # trading cost per trade, in basis points of the amount traded

LOOKBACK_GRID = [3, 5, 7, 10]   # section 5: lookbacks to compare
MIN_AVG_GRID = [0.0, 0.01, 0.02]  # section 5: thresholds for condition 1
SPLIT = "2017Q1"                # section 6: choose settings before this quarter, test from it onwards
N_SIMULATIONS = 2000            # section 7: random portfolios in the luck test

In [ ]:
# Backtest engine. Everything the notebook needs is defined in this cell; nothing is imported from other files.
import math
import warnings
from dataclasses import dataclass

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

RULE, SEASONAL, SPY_LABEL, EW = "Seasonal dip-buy", "Seasonal only", "S&P 500 (SPY)", "Equal-weight sectors"


# ------------------------------------------------------------------------------------ data

def load_prices(tickers, start, end=None):
    """Daily closes from Yahoo Finance, adjusted for splits and dividends. One column per ticker."""
    import yfinance as yf

    tickers = list(dict.fromkeys(tickers))
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False, group_by="column", threads=True)
    if raw is None or raw.empty:
        raise RuntimeError("Yahoo Finance returned no data. Check the internet connection, wait a minute if "
                           "rate-limited, or update yfinance with: pip install -U yfinance")
    if isinstance(raw.columns, pd.MultiIndex):
        top = raw.columns.get_level_values(0)
        if "Close" in top or "Adj Close" in top:
            close = raw["Close" if "Close" in top else "Adj Close"]
        else:
            close = raw.xs("Close", axis=1, level=1)
    else:
        close = raw[["Close"]].rename(columns={"Close": tickers[0]})
    if isinstance(close, pd.Series):
        close = close.to_frame(tickers[0])
    close = close.reindex(columns=tickers).astype(float)
    idx = pd.to_datetime(close.index)
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_localize(None)
    close.index = idx.normalize()
    close = close[~close.index.duplicated(keep="last")].sort_index().dropna(how="all")
    missing = [t for t in tickers if close[t].dropna().empty]
    if missing:
        warnings.warn(f"no data returned for {missing}")
    return close


def quarter_calendar(index, lag=1):
    """One row per completed calendar quarter: its last trading day and the rebalance day `lag` trading days later.

    The quarter in progress is left out until the data reaches its last business day.
    """
    idx = pd.DatetimeIndex(index).sort_values().unique()
    last_day = pd.Series(idx, index=idx).groupby(idx.to_period("Q")).max()
    last_bday = pd.offsets.BDay().rollback(last_day.index[-1].end_time.normalize())
    if idx[-1] < last_bday:
        last_day = last_day.iloc[:-1]
    pos = idx.get_indexer(pd.DatetimeIndex(last_day.values)) + lag
    rebalance = [idx[p] if p < len(idx) else pd.NaT for p in pos]
    cal = pd.DataFrame({"quarter_end": pd.DatetimeIndex(last_day.values), "rebalance": pd.DatetimeIndex(rebalance)},
                       index=last_day.index)
    cal.index.name = "quarter"
    return cal


def _prices_on(px, dates, index):
    p = px.ffill(limit=5).reindex(pd.DatetimeIndex(dates))
    p.index = index
    return p.where(p > 0)


def quarterly_returns(px, cal):
    """Calendar-quarter returns, close of the last trading day to close of the last trading day."""
    p = _prices_on(px, cal["quarter_end"].values, cal.index)
    return p / p.shift(1) - 1


def holding_returns(px, cal):
    """Return in quarter q of a position bought on the rebalance day after quarter q-1 and sold on the one after q."""
    p = _prices_on(px, cal["rebalance"].values, cal.index)
    return p / p.shift(1) - 1


def cash_returns(irx, cal):
    """Quarterly T-bill return: the 13-week yield (^IRX, in percent) on the rebalance day that starts the quarter."""
    if irx is None or irx.dropna().empty:
        return pd.Series(0.0, index=cal.index)
    y = irx.ffill().reindex(pd.DatetimeIndex(cal["rebalance"].values))
    y.index = cal.index
    return (y.shift(1) / 100 / 4).fillna(0.0).clip(lower=0.0)


def seasonal_average(qret, years):
    """For quarter q: the mean return of the same calendar quarter in the `years` years before q.

    Uses only quarters q-4, q-8, ..., q-4*years, so it is known before q starts. NaN without all `years` values.
    """
    stack = np.stack([qret.shift(4 * k).to_numpy(dtype=float) for k in range(1, years + 1)])
    count = np.isfinite(stack).sum(axis=0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        avg = np.nanmean(stack, axis=0)
    avg[count < years] = np.nan
    return pd.DataFrame(avg, index=qret.index, columns=qret.columns)


# ------------------------------------------------------------------------------------ backtest

def _simulate(weights, hold, cash, cost_bps):
    """Net quarterly returns of a weight schedule (rows sum to 1, or 0 for all cash) with trading costs."""
    cols = list(weights.columns)
    prev = pd.Series(0.0, index=cols)
    net, traded_s, cost_s = {}, {}, {}
    for q in weights.index:
        w = weights.loc[q]
        r = hold.loc[q, cols].fillna(0.0)
        cash_w = 1.0 - float(w.sum())
        traded = float((w - prev).abs().sum())
        cost = traded * cost_bps / 1e4
        gross = float((w * r).sum()) + cash_w * float(cash.loc[q])
        net[q], traded_s[q], cost_s[q] = gross - cost, traded, cost
        grown = w * (1 + r)
        total = float(grown.sum()) + cash_w * (1 + float(cash.loc[q]))
        prev = grown / total if total > 0 else w * 0.0
    return pd.Series(net), pd.Series(traded_s), pd.Series(cost_s)


@dataclass
class Backtest:
    returns: pd.DataFrame      # net quarterly returns, one column per strategy, indexed by the quarter held
    cash: pd.Series            # T-bill return per quarter
    weights: dict              # strategy -> weights DataFrame
    turnover: dict             # strategy -> traded fraction of the portfolio per quarter
    costs: dict                # strategy -> cost per quarter
    seasonal_avg: pd.DataFrame # next-quarter seasonal average known at the start of each quarter
    prior_return: pd.DataFrame # return of the quarter just finished, at the start of each quarter
    hold: pd.DataFrame         # realised return of each ETF over each holding quarter
    eligible: pd.DataFrame     # sectors with a signal at the start of each quarter
    names: dict                # ticker -> sector name
    params: dict

    @property
    def sectors(self):
        return list(self.names)

    def equity(self, cols=None):
        r = self.returns[cols] if cols is not None else self.returns
        eq = (1 + r).cumprod()
        start = pd.DataFrame([[1.0] * eq.shape[1]], columns=eq.columns, index=[eq.index[0] - 1])
        return pd.concat([start, eq])

    def holdings(self):
        w = self.weights[RULE][self.sectors]
        idle = f"nothing qualified, held {self.params['empty_quarter']}"
        return w.apply(lambda row: ", ".join(self.names[t] for t in row.index[row > 0]) or idle, axis=1)

    def trades(self):
        """One row per sector bought: the quarter held, the signal that triggered it and what it returned."""
        w = self.weights[RULE][self.sectors]
        rows = []
        for q in w.index:
            for t in w.columns[w.loc[q] > 0]:
                rows.append({"quarter": q, "sector": self.names[t], "ticker": t,
                             "prior_quarter_return": self.prior_return.loc[q, t],
                             "seasonal_average": self.seasonal_avg.loc[q, t],
                             "realised_return": self.hold.loc[q, t]})
        return pd.DataFrame(rows)


def _apply_empty(w, empty, eligible, benchmark):
    w = w.copy()
    w[benchmark] = 0.0
    idle = w.sum(axis=1) == 0
    if empty == "spy":
        w.loc[idle, benchmark] = 1.0
    elif empty == "equal_weight":
        ew = eligible.astype(float)
        ew = ew.div(ew.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
        w.loc[idle, ew.columns] = ew.loc[idle].to_numpy()
    elif empty != "cash":
        raise ValueError("empty_quarter must be 'cash', 'spy' or 'equal_weight'")
    return w


def run_backtest(px, cal, sectors, lookback, min_avg=0.0, margin=0.0, empty_quarter="cash", cost_bps=5.0,
                 benchmark="SPY", cash=None, min_universe=5):
    """Backtest the seasonal dip-buy rule and its comparisons.

    At the start of each quarter q, a sector is bought when
      1. its average return in q's calendar quarter over the previous `lookback` years is above `min_avg`, and
      2. its return in the quarter just finished is below that average by more than `margin`.
    Qualifying sectors are held equally weighted for the quarter. Quarters where nothing qualifies go to
    `empty_quarter`. Only quarters with at least `min_universe` sectors that have a signal are evaluated.
    """
    names = {t: n for n, t in sectors.items() if t in px.columns and px[t].notna().any()}
    tick = list(names)
    cash = cash if cash is not None else pd.Series(0.0, index=cal.index)
    qret = quarterly_returns(px[tick], cal)
    hold = holding_returns(px[tick + [benchmark]], cal)
    avg = seasonal_average(qret, lookback)
    prior = qret.shift(1)
    alive = prior.notna()
    eligible = avg.notna() & alive
    quarters = eligible.index[(eligible.sum(axis=1) >= min_universe) & hold[benchmark].notna()]
    if len(quarters) == 0:
        raise ValueError(f"no quarter has {min_universe} sectors with {lookback} years of history; start earlier")

    def weights_for(pick):
        k = pick.sum(axis=1)
        return pick.astype(float).div(k.replace(0, np.nan), axis=0).fillna(0.0)

    seasonal_pick = eligible & (avg > min_avg)
    rule_pick = seasonal_pick & (prior < avg - margin)
    w = {
        RULE: _apply_empty(weights_for(rule_pick), empty_quarter, eligible, benchmark),
        SEASONAL: _apply_empty(weights_for(seasonal_pick), empty_quarter, eligible, benchmark),
        SPY_LABEL: pd.DataFrame(0.0, index=cal.index, columns=tick + [benchmark]).assign(**{benchmark: 1.0}),
        EW: _apply_empty(weights_for(alive), "cash", alive, benchmark),
    }
    out, turn, costs = {}, {}, {}
    for label, wt in w.items():
        wt = wt.loc[quarters]
        w[label] = wt
        out[label], turn[label], costs[label] = _simulate(wt, hold, cash, cost_bps)
    returns = pd.DataFrame(out)[[RULE, SEASONAL, SPY_LABEL, EW]]
    params = dict(lookback=lookback, min_avg=min_avg, margin=margin, empty_quarter=empty_quarter,
                  cost_bps=cost_bps, benchmark=benchmark, lag=None)
    return Backtest(returns=returns, cash=cash.loc[quarters], weights=w, turnover=turn, costs=costs,
                    seasonal_avg=avg.loc[quarters], prior_return=prior.loc[quarters], hold=hold.loc[quarters],
                    eligible=eligible.loc[quarters], names=names, params=params)


# ------------------------------------------------------------------------------------ statistics

def performance(bt, quarters=None):
    """Headline statistics for every strategy in a backtest, optionally over a subset of quarters."""
    rets = bt.returns if quarters is None else bt.returns.loc[quarters]
    cash = bt.cash.loc[rets.index]
    spy = rets[SPY_LABEL]
    n = len(rets)
    rows = {}
    for col in rets.columns:
        r = rets[col]
        eq = pd.concat([pd.Series([1.0]), (1 + r).cumprod().reset_index(drop=True)])
        cagr = eq.iloc[-1] ** (4 / n) - 1
        ex = r - cash
        down = math.sqrt(float((np.minimum(ex, 0) ** 2).mean()))
        mdd = float((eq / eq.cummax() - 1).min())
        diff = r - spy
        row = {
            "CAGR": cagr,
            "Volatility": r.std(ddof=1) * 2,
            "Sharpe": ex.mean() / ex.std(ddof=1) * 2 if ex.std(ddof=1) > 0 else np.nan,
            "Sortino": ex.mean() * 4 / (down * 2) if down > 0 else np.nan,
            "Max drawdown": mdd,
            "Calmar": cagr / abs(mdd) if mdd < 0 else np.nan,
            "Best quarter": r.max(),
            "Worst quarter": r.min(),
            "Quarters positive": (r > 0).mean(),
            "Quarters beating SPY": (diff > 0).mean() if col != SPY_LABEL else np.nan,
            "CAGR vs SPY": cagr - ((1 + spy).prod() ** (4 / n) - 1) if col != SPY_LABEL else np.nan,
            "t-stat vs SPY": diff.mean() / diff.std(ddof=1) * math.sqrt(n) if col != SPY_LABEL and diff.std() > 0 else np.nan,
        }
        wt = bt.weights[col].loc[rets.index, bt.sectors]
        held = (wt > 0).sum(axis=1)
        row["Quarters invested"] = (held > 0).mean() if col in (RULE, SEASONAL) else np.nan
        row["Avg sectors held"] = held[held > 0].mean() if col in (RULE, SEASONAL, EW) else np.nan
        row["Turnover / year"] = bt.turnover[col].loc[rets.index].mean() * 4 / 2
        rows[col] = row
    out = pd.DataFrame(rows)
    out.attrs["period"] = f"{rets.index[0]} to {rets.index[-1]} ({n} quarters)"
    return out


def calendar_years(bt, cols=(RULE, SPY_LABEL)):
    r = bt.returns[list(cols)]
    by_year = (1 + r).groupby(r.index.year).prod() - 1
    by_year.index.name = "year"
    return by_year


def sector_breakdown(bt):
    """Per sector: how often the rule bought it, its average return when bought and the share of winning trades."""
    tr = bt.trades()
    if tr.empty:
        return pd.DataFrame(columns=["times_bought", "avg_return_when_bought", "win_rate", "avg_return_all_quarters"])
    g = tr.groupby("sector")["realised_return"]
    out = pd.DataFrame({"times_bought": g.size(), "avg_return_when_bought": g.mean(), "win_rate": g.apply(lambda s: (s > 0).mean())})
    all_q = bt.hold[bt.sectors].rename(columns=bt.names).where(bt.eligible.rename(columns=bt.names))
    out["avg_return_all_quarters"] = all_q.mean().reindex(out.index)
    return out.sort_values("avg_return_when_bought", ascending=False)


def filter_effect(bt):
    """Among sectors that passed the seasonal test, compare the ones the dip filter bought with the ones it skipped."""
    p = bt.params
    seasonal = bt.eligible & (bt.seasonal_avg > p["min_avg"])
    bought = seasonal & (bt.prior_return < bt.seasonal_avg - p["margin"])
    skipped = seasonal & ~bought
    r = bt.hold[bt.sectors]
    rows = {}
    for label, mask in [("Bought: finished the prior quarter below the average", bought),
                        ("Skipped: finished the prior quarter at or above it", skipped)]:
        vals = r.where(mask).stack().dropna()
        rows[label] = {"sector_quarters": int(vals.size), "avg_return": vals.mean(), "median_return": vals.median(),
                       "win_rate": (vals > 0).mean() if vals.size else np.nan}
    return pd.DataFrame(rows).T


def setting_labels(df):
    """Replace the (lookback, min_avg) index with labels like '5 years · avg > 1%'."""
    out = df.copy()
    out.index = [f"{int(L)} years · avg > {m:.0%}" for L, m in df.index]
    out.index.name = "setting"
    return out


def fmt(df, **formats):
    """Display a table with the given column formats, e.g. fmt(df, CAGR="{:.1%}")."""
    return df.style.format(formats, na_rep="–")


_RATIO_ROWS = {"Sharpe", "Sortino", "Calmar", "t-stat vs SPY", "Avg sectors held"}
_WHOLE_PCT_ROWS = {"Quarters positive", "Quarters beating SPY", "Quarters invested", "Turnover / year"}


def fmt_performance(p):
    """Display the performance table with a format per row."""
    sty = p.style
    for row in p.index:
        spec = ("{:.2f}" if row in _RATIO_ROWS else "{:.0%}" if row in _WHOLE_PCT_ROWS
                else "{:+.1%}" if row == "CAGR vs SPY" else "{:.1%}")
        sty = sty.format(spec, subset=pd.IndexSlice[[row], :], na_rep="–")
    return sty.set_caption(p.attrs.get("period", ""))


def grid_search(px, cal, sectors, lookbacks, min_avgs, **kw):
    """Run every (lookback, minimum average) pair and compare them over the quarters they all cover."""
    bts = {(L, m): run_backtest(px, cal, sectors, L, min_avg=m, **kw) for L in lookbacks for m in min_avgs}
    common = None
    for bt in bts.values():
        common = bt.returns.index if common is None else common.intersection(bt.returns.index)
    rows = {}
    for key, bt in bts.items():
        p = performance(bt, common)
        rows[key] = p[RULE]
    table = pd.DataFrame(rows).T
    table.index.names = ["lookback_years", "min_avg"]
    spy = performance(next(iter(bts.values())), common)[SPY_LABEL]
    table.attrs.update(period=f"{common[0]} to {common[-1]} ({len(common)} quarters)", spy=spy, common=common)
    return table, bts


def in_and_out_of_sample(bts, split, metric="Sharpe"):
    """Pick the best configuration on quarters before `split`, then report every configuration after it."""
    split = pd.Period(split, freq="Q")
    common = None
    for bt in bts.values():
        common = bt.returns.index if common is None else common.intersection(bt.returns.index)
    ins, oos = common[common < split], common[common >= split]
    if len(ins) < 8 or len(oos) < 8:
        raise ValueError(f"split {split} leaves too few quarters on one side ({len(ins)} in, {len(oos)} out)")
    rows = {}
    for key, bt in bts.items():
        a, b = performance(bt, ins), performance(bt, oos)
        rows[key] = {"in-sample " + metric: a.loc[metric, RULE], "out-of-sample " + metric: b.loc[metric, RULE],
                     "in-sample CAGR": a.loc["CAGR", RULE], "out-of-sample CAGR": b.loc["CAGR", RULE],
                     "out-of-sample CAGR vs SPY": b.loc["CAGR vs SPY", RULE]}
    table = pd.DataFrame(rows).T
    table.index.names = ["lookback_years", "min_avg"]
    best = table["in-sample " + metric].idxmax()
    spy_oos = performance(next(iter(bts.values())), oos)[SPY_LABEL]
    table.attrs.update(best=best, spy_oos=spy_oos, in_period=f"{ins[0]} to {ins[-1]}", out_period=f"{oos[0]} to {oos[-1]}")
    return table.sort_values("in-sample " + metric, ascending=False)


def random_pick_test(bt, n_sims=2000, seed=7):
    """Luck test: each quarter, hold the same number of sectors as the rule did, chosen at random.

    Returns the simulated CAGRs and the share that matched or beat the rule. Quarters where the rule held
    nothing keep the rule's own return, and each quarter's trading cost is carried over.
    """
    rng = np.random.default_rng(seed)
    w = bt.weights[RULE][bt.sectors]
    k = (w > 0).sum(axis=1)
    growth = np.ones(n_sims)
    for q in bt.returns.index:
        rule_r = float(bt.returns.loc[q, RULE])
        if k.loc[q] == 0:
            growth *= 1 + rule_r
            continue
        elig = bt.eligible.loc[q, bt.sectors]
        r = bt.hold.loc[q, bt.sectors][elig].to_numpy(dtype=float)
        pick = np.argsort(rng.random((n_sims, len(r))), axis=1)[:, : int(k.loc[q])]
        growth *= 1 + r[pick].mean(axis=1) - float(bt.costs[RULE].loc[q])
    sims = growth ** (4 / len(bt.returns)) - 1
    actual = performance(bt).loc["CAGR", RULE]
    return sims, float((sims >= actual).mean())


def current_signal(px, cal, sectors, lookback, min_avg=0.0, margin=0.0):
    """What the rule says right now for the next quarter, using the quarter in progress as the 'current' quarter."""
    tick = [t for t in sectors.values() if t in px.columns]
    names = {t: n for n, t in sectors.items() if t in tick}
    qret = quarterly_returns(px[tick], cal)
    last_q, last_close = cal.index[-1], cal["quarter_end"].iloc[-1]
    latest = px.index[-1]
    if latest > last_close:
        current_q, next_q = last_q + 1, last_q + 2
        cur = px[tick].ffill().iloc[-1] / px[tick].ffill().loc[last_close] - 1
        cur_label = f"{current_q} to date ({latest:%b %d})"
    else:
        current_q, next_q = last_q, last_q + 1
        cur = qret.iloc[-1]
        cur_label = f"{current_q} (complete)"
    same = qret[qret.index.quarter == next_q.quarter].tail(lookback)
    avg = same.mean().where(same.notna().sum() == lookback)
    out = pd.DataFrame({"ticker": tick, "current_quarter": cur.reindex(tick).to_numpy(),
                        "next_quarter_average": avg.reindex(tick).to_numpy()}, index=[names[t] for t in tick])
    out["qualifies"] = (out["next_quarter_average"] > min_avg) & (out["current_quarter"] < out["next_quarter_average"] - margin)
    out["gap"] = out["current_quarter"] - out["next_quarter_average"]
    years = sorted({p.year for p in same.index})
    out.attrs.update(current=cur_label, next=str(next_q), years=f"{years[0]}–{years[-1]}" if years else "n/a")
    return out.sort_values(["qualifies", "gap"], ascending=[False, True])


def scorecard(bt, grid, oos, mc_p):
    """Plain pass / fail checks on the evidence, each against a stated bar."""
    p = performance(bt)
    best = oos.attrs["best"]
    row_oos = oos.loc[best]
    checks = [
        ("Higher CAGR than SPY", f"{p.loc['CAGR', RULE]:.1%} vs {p.loc['CAGR', SPY_LABEL]:.1%}",
         p.loc["CAGR", RULE] > p.loc["CAGR", SPY_LABEL]),
        ("Higher Sharpe ratio than SPY", f"{p.loc['Sharpe', RULE]:.2f} vs {p.loc['Sharpe', SPY_LABEL]:.2f}",
         p.loc["Sharpe", RULE] > p.loc["Sharpe", SPY_LABEL]),
        ("Smaller maximum drawdown than SPY", f"{p.loc['Max drawdown', RULE]:.1%} vs {p.loc['Max drawdown', SPY_LABEL]:.1%}",
         p.loc["Max drawdown", RULE] > p.loc["Max drawdown", SPY_LABEL]),
        ("Quarterly edge over SPY is statistically significant (t > 2)", f"t = {p.loc['t-stat vs SPY', RULE]:.2f}",
         p.loc["t-stat vs SPY", RULE] > 2),
        ("The dip filter adds return over seasonal-only", f"{p.loc['CAGR', RULE]:.1%} vs {p.loc['CAGR', SEASONAL]:.1%}",
         p.loc["CAGR", RULE] > p.loc["CAGR", SEASONAL]),
        ("Best in-sample setting also beats SPY out of sample",
         f"lookback {best[0]}y, min avg {best[1]:.0%}: {row_oos['out-of-sample CAGR vs SPY']:+.1%} a year",
         row_oos["out-of-sample CAGR vs SPY"] > 0),
        ("Beats random sector picks at the 5% level", f"p = {mc_p:.3f}", mc_p < 0.05),
    ]
    return pd.DataFrame(checks, columns=["check", "result", "pass"]).assign(
        **{"pass": lambda d: d["pass"].map({True: "✔ pass", False: "✘ fail"})}).set_index("check")


# ------------------------------------------------------------------------------------ charts

BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
NAVY, INK, INK2, MUTED, GRID, RED = "#14285a", "#1a1a1a", "#52514e", "#898781", "#e1e0d9", "#e34948"
LINE_COLORS = {RULE: BLUE, SPY_LABEL: ORANGE, SEASONAL: AQUA, EW: YELLOW}


def set_style():
    plt.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 150, "figure.facecolor": "white", "axes.facecolor": "white",
        "font.family": "sans-serif", "font.size": 10.5, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
        "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
        "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK2, "ytick.color": INK2,
        "legend.frameon": False, "axes.titlelocation": "left",
    })


def _titles(ax, title, subtitle=None):
    ax.set_title(title, loc="left", fontsize=14, fontweight="bold", color=INK, pad=26 if subtitle else 10)
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, fontsize=10, color=INK2, va="bottom")


def _end_labels(ax, series_by_label, fmt, log=False):
    """Direct labels at the right end of each line, nudged apart so they don't overlap."""
    ends = sorted(((float(s.iloc[-1]), lab) for lab, s in series_by_label.items()), key=lambda x: x[0])
    lo, hi = ax.get_ylim()
    to_u = (lambda v: math.log(v)) if log else (lambda v: v)
    from_u = (lambda u: math.exp(u)) if log else (lambda u: u)
    gap = (to_u(hi) - to_u(lo)) * 0.055
    placed = []
    for v, lab in ends:
        u = to_u(v)
        if placed and u - placed[-1] < gap:
            u = placed[-1] + gap
        placed.append(u)
        ax.annotate(f"{lab}  {fmt(v)}", xy=(1.005, from_u(u)), xycoords=("axes fraction", "data"),
                    color=LINE_COLORS.get(lab, INK2), fontsize=9.5, va="center", annotation_clip=False)


def equity_chart(bt, cols=(RULE, SEASONAL, EW, SPY_LABEL), log=True):
    eq = bt.equity(list(cols))
    x = eq.index.to_timestamp(how="end")
    fig, ax = plt.subplots(figsize=(11, 5.2))
    for col in cols:
        ax.plot(x, eq[col], color=LINE_COLORS[col], lw=2.2 if col == RULE else 1.6, label=col)
    if log:
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(mticker.LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}" if v >= 1 else f"${v:.2f}"))
        ax.yaxis.set_minor_locator(mticker.NullLocator())
    ax.set_xlim(x[0], x[-1])
    _end_labels(ax, {c: eq[c] for c in cols}, lambda v: f"${v:,.2f}", log=log)
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.08), ncol=4, fontsize=9.5)
    _titles(ax, "Growth of $1", f"{bt.returns.index[0]} to {bt.returns.index[-1]} · net of {bt.params['cost_bps']:g} bps "
            f"trading costs · {'log' if log else 'linear'} scale")
    fig.subplots_adjust(right=0.78)
    return fig


def drawdown_chart(bt):
    eq = bt.equity([RULE, SPY_LABEL])
    dd = eq / eq.cummax() - 1
    x = dd.index.to_timestamp(how="end")
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.fill_between(x, dd[RULE], 0, color=BLUE, alpha=0.25, lw=0)
    ax.plot(x, dd[RULE], color=BLUE, lw=1.8, label=RULE)
    ax.plot(x, dd[SPY_LABEL], color=ORANGE, lw=1.4, label=SPY_LABEL)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_xlim(x[0], x[-1])
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.14), ncol=2)
    _titles(ax, "Drawdowns", "Decline from the previous peak, measured at quarter ends")
    return fig


def yearly_chart(bt):
    yr = calendar_years(bt)
    fig, ax = plt.subplots(figsize=(11, 4.2))
    xs = np.arange(len(yr))
    ax.bar(xs - 0.2, yr[RULE], width=0.38, color=BLUE, label=RULE)
    ax.bar(xs + 0.2, yr[SPY_LABEL], width=0.38, color=ORANGE, label=SPY_LABEL)
    ax.axhline(0, color=INK2, lw=0.8)
    ax.set_xticks(xs, [str(y) for y in yr.index], rotation=0 if len(yr) < 16 else 45)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
    _titles(ax, "Calendar-year returns", "Compounded from quarterly returns; the first and last years can be partial")
    return fig


def bar_chart(values, title, subtitle=None, fmt="{:.1%}", color=NAVY):
    """Horizontal bars sorted highest to lowest, value labels at the bar ends (the dashboard's style)."""
    v = values.dropna().sort_values()
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(v) + 1.4))
    ax.barh(range(len(v)), v.to_numpy(), color=color, height=0.55)
    ax.set_yticks(range(len(v)), [str(i) for i in v.index])
    span = max(v.max(), 0) - min(v.min(), 0) or 1
    for i, val in enumerate(v):
        ax.text(val + (0.012 * span if val >= 0 else -0.012 * span), i, fmt.format(val), va="center",
                ha="left" if val >= 0 else "right", fontsize=9.5, color=INK)
    lo, hi = min(v.min(), 0), max(v.max(), 0)
    ax.set_xlim(lo - 0.2 * span if lo < 0 else 0, hi + 0.2 * span if hi > 0 else 0)
    ax.axvline(0, color=INK, lw=0.9)
    ax.grid(False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0, labelcolor=INK)
    if "%" in fmt:
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    _titles(ax, title, subtitle)
    fig.tight_layout()
    return fig


_DIVERGING = LinearSegmentedColormap.from_list("better_worse", [RED, "#f0efec", BLUE])


def heatmap(grid, value, title, subtitle, center, fmt="{:.2f}"):
    """Lookback (rows) by minimum average (columns); blue is better than `center`, red is worse."""
    m = grid[value].unstack("min_avg")
    vals = m.to_numpy(dtype=float)
    span = max(np.nanmax(np.abs(vals - center)), 1e-9)
    norm = TwoSlopeNorm(vcenter=center, vmin=center - span, vmax=center + span)
    fig, ax = plt.subplots(figsize=(1.6 * m.shape[1] + 3, 0.7 * m.shape[0] + 1.8))
    ax.imshow(vals, cmap=_DIVERGING, norm=norm, aspect="auto")
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, fmt.format(vals[i, j]), ha="center", va="center", fontsize=11,
                    color="white" if abs(vals[i, j] - center) > 0.6 * span else INK)
    ax.set_xticks(range(m.shape[1]), [f"avg > {c:.0%}" for c in m.columns])
    ax.set_yticks(range(m.shape[0]), [f"{r} years" for r in m.index])
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)
    _titles(ax, title, subtitle)
    fig.tight_layout()
    return fig


def luck_chart(sims, actual, spy_cagr, p):
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.hist(sims, bins=50, color="#86b6ef", edgecolor="white", linewidth=0.5)
    ax.axvline(actual, color=BLUE, lw=2.2)
    ax.axvline(spy_cagr, color=ORANGE, lw=1.6, ls="--")
    top = ax.get_ylim()[1]
    ax.text(actual, top * 0.95, f"  {RULE} {actual:.1%}", color=BLUE, va="top", fontsize=9.5)
    ax.text(spy_cagr, top * 0.80, f"  SPY {spy_cagr:.1%}", color=ORANGE, va="top", fontsize=9.5)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_ylabel("simulations")
    _titles(ax, "Could random picks have done as well?",
            f"CAGR of {len(sims):,} portfolios holding the same number of random sectors each quarter · "
            f"{p:.1%} matched or beat the rule")
    return fig

In [ ]:
from IPython.display import Markdown, display

set_style()
prices = load_prices(list(SECTORS.values()) + [BENCHMARK], start=START)
try:
    irx = load_prices([CASH_TICKER], start=START)[CASH_TICKER]
except Exception as exc:  # the backtest still runs, with idle cash earning 0%
    print(f"T-bill yields unavailable ({exc}); idle cash earns 0%")
    irx = None

cal = quarter_calendar(prices.index, lag=EXECUTION_LAG_DAYS)
cash = cash_returns(irx, cal)
common = dict(empty_quarter=EMPTY_QUARTER, cost_bps=COST_BPS, benchmark=BENCHMARK, cash=cash, margin=DIP_MARGIN)
bt = run_backtest(prices, cal, SECTORS, LOOKBACK_YEARS, min_avg=MIN_SEASONAL_AVG, **common)

coverage = pd.DataFrame({
    "ticker": list(SECTORS.values()) + [BENCHMARK],
    "first_day": [prices[t].first_valid_index() for t in list(SECTORS.values()) + [BENCHMARK]],
    "last_day": [prices[t].last_valid_index() for t in list(SECTORS.values()) + [BENCHMARK]],
}, index=list(SECTORS) + ["S&P 500"])
print(f"Prices {prices.index[0]:%Y-%m-%d} to {prices.index[-1]:%Y-%m-%d} · last complete quarter {cal.index[-1]}")
print(f"Backtest with a {LOOKBACK_YEARS}-year lookback covers {bt.returns.index[0]} to {bt.returns.index[-1]} "
      f"({len(bt.returns)} quarters)")
fmt(coverage, first_day="{:%Y-%m-%d}", last_day="{:%Y-%m-%d}")

## 1. How the rule works: the most recent quarter

The decision made at the start of the last completed quarter, sector by sector. **Prior quarter** is the return of
the quarter that had just ended. **Seasonal average** is the average return of the quarter being bought, over the
previous years. A sector is bought when the seasonal average is above the threshold and the prior quarter is
below it.

In [ ]:
q = bt.returns.index[-1]
decision = pd.DataFrame({
    "ticker": bt.sectors,
    "prior_quarter": bt.prior_return.loc[q, bt.sectors].to_numpy(),
    "seasonal_average": bt.seasonal_avg.loc[q, bt.sectors].to_numpy(),
    "bought": (bt.weights[RULE].loc[q, bt.sectors] > 0).map({True: "✔ bought", False: ""}).to_numpy(),
    "realised_return": bt.hold.loc[q, bt.sectors].to_numpy(),
}, index=[bt.names[t] for t in bt.sectors]).sort_values(["bought", "seasonal_average"], ascending=[False, False])
display(Markdown(f"**Quarter held: {q}.** Prior quarter {q - 1}; seasonal average of Q{q.quarter} over the "
                 f"{LOOKBACK_YEARS} years before. Rule return for the quarter: **{bt.returns.loc[q, RULE]:+.2%}**, "
                 f"S&P 500 **{bt.returns.loc[q, SPY_LABEL]:+.2%}**."))
fmt(decision, prior_quarter="{:+.1%}", seasonal_average="{:+.1%}", realised_return="{:+.1%}")

## 2. Performance

Four strategies over the same quarters, all net of trading costs:

* **Seasonal dip-buy** is your rule.
* **Seasonal only** buys every sector with a positive seasonal average, without the dip filter, so the two can be
  compared.
* **Equal-weight sectors** holds every sector, rebalanced each quarter.
* **S&P 500 (SPY)** is buy and hold.

The Sharpe and Sortino ratios are measured against T-bills. The t-stat tests whether the rule's quarterly
return beat SPY's by more than chance; above about 2 is the usual bar.

In [ ]:
perf = performance(bt)
fmt_performance(perf)

In [ ]:
equity_chart(bt)
plt.show()
drawdown_chart(bt)
plt.show()

In [ ]:
yearly_chart(bt)
plt.show()
years = calendar_years(bt, cols=(RULE, SEASONAL, SPY_LABEL))
years["rule_minus_spy"] = years[RULE] - years[SPY_LABEL]
fmt(years, **{c: "{:+.1%}" for c in years.columns})

## 3. What the rule traded

Every sector-quarter the rule bought, grouped by sector. **Average return when bought** is the realised return over
the quarter held; **average return, all quarters** is the same sector over every quarter in the test, for
comparison. A rule with an edge buys sectors in quarters that beat their own usual return.

In [ ]:
breakdown = sector_breakdown(bt)
bar_chart(breakdown["times_bought"], "Times each sector was bought",
          f"Quarters held out of {len(bt.returns)} · lookback {LOOKBACK_YEARS} years", fmt="{:.0f}")
plt.show()
bar_chart(breakdown["avg_return_when_bought"], "Average return when bought",
          "Realised return in the quarter held, by sector")
plt.show()
fmt(breakdown, avg_return_when_bought="{:+.2%}", win_rate="{:.0%}", avg_return_all_quarters="{:+.2%}")

In [ ]:
trades = bt.trades()
print(f"{len(trades)} trades · win rate {(trades['realised_return'] > 0).mean():.0%} · "
      f"average {trades['realised_return'].mean():+.2%} per trade-quarter")
log = pd.DataFrame({"held": bt.holdings(), "rule": bt.returns[RULE], "spy": bt.returns[SPY_LABEL]})
log["rule_minus_spy"] = log["rule"] - log["spy"]
display(Markdown("**Last 12 quarters**"))
fmt(log.tail(12), rule="{:+.1%}", spy="{:+.1%}", rule_minus_spy="{:+.1%}")

## 4. Does the dip filter help?

Condition 2 is what makes this rule yours. The seasonal part on its own is the well-studied "return seasonality"
effect. The test below looks only at sectors that passed condition 1, and splits them by condition 2. If the
filter adds value, the sectors it bought should have done better than the ones it skipped.

Research on sectors and commodities has generally found that recent winners keep winning in the short run, which
is the opposite of what the filter assumes. This is the part of the idea most worth checking.

In [ ]:
fe = filter_effect(bt)
display(fmt(fe, sector_quarters="{:.0f}", avg_return="{:+.2%}", median_return="{:+.2%}", win_rate="{:.0%}"))
comparison = perf.loc[["CAGR", "Sharpe", "Max drawdown", "Quarters invested", "Avg sectors held"], [RULE, SEASONAL]]
fmt_performance(comparison)

## 5. Choosing the lookback and threshold

Every combination of lookback (years averaged) and minimum seasonal average, compared over **the same quarters**.
A 10-year lookback needs 10 years of history before its first trade, so all settings are measured from that point
on. In the heatmaps, **blue beats SPY and red trails it**; the darker the colour, the bigger the gap.

Testing many settings and picking the best inflates results. Section 6 checks for that.

In [ ]:
grid, grid_bts = grid_search(prices, cal, SECTORS, LOOKBACK_GRID, MIN_AVG_GRID, **common)
spy_ref = grid.attrs["spy"]
heatmap(grid, "Sharpe", "Sharpe ratio by setting",
        f"{grid.attrs['period']} · SPY = {spy_ref['Sharpe']:.2f}", center=spy_ref["Sharpe"])
plt.show()
heatmap(grid, "CAGR vs SPY", "Annual return versus SPY by setting", grid.attrs["period"], center=0.0, fmt="{:+.1%}")
plt.show()
cols = ["CAGR", "CAGR vs SPY", "Sharpe", "Max drawdown", "t-stat vs SPY", "Quarters invested", "Avg sectors held"]
fmt(setting_labels(grid[cols].sort_values("Sharpe", ascending=False)), CAGR="{:.1%}", **{"CAGR vs SPY": "{:+.1%}"},
    Sharpe="{:.2f}", **{"Max drawdown": "{:.1%}", "t-stat vs SPY": "{:.2f}", "Quarters invested": "{:.0%}",
                        "Avg sectors held": "{:.1f}"})

## 6. Out-of-sample check

The honest version of section 5. Pick the setting with the best Sharpe ratio using **only the quarters before
`SPLIT`**, then see how that same setting did **afterwards**, on quarters it never saw. A setting that was only
lucky tends to fall apart here.

In [ ]:
oos = in_and_out_of_sample(grid_bts, SPLIT)
best = oos.attrs["best"]
spy_oos = oos.attrs["spy_oos"]
display(Markdown(
    f"In-sample {oos.attrs['in_period']}, out-of-sample {oos.attrs['out_period']}.  \n"
    f"Best in-sample setting: **{best[0]}-year lookback, average above {best[1]:.0%}**. Out of sample it returned "
    f"**{oos.loc[best, 'out-of-sample CAGR']:.1%} a year** against SPY's **{spy_oos['CAGR']:.1%}**, with a Sharpe "
    f"ratio of **{oos.loc[best, 'out-of-sample Sharpe']:.2f}** against SPY's **{spy_oos['Sharpe']:.2f}**."))
fmt(setting_labels(oos), **{c: "{:.2f}" for c in oos.columns if "Sharpe" in c},
    **{c: "{:+.1%}" for c in oos.columns if "CAGR" in c})

## 7. Luck test

Would picking sectors at random have done as well? Each simulated portfolio holds **the same number of sectors as
the rule, every quarter**, but chooses them at random from the sectors available that quarter. It pays the same
trading costs. If many random portfolios match the rule, its result could be luck. The p-value is the share that
matched or beat it; below 0.05 is the usual bar.

In [ ]:
sims, p_value = random_pick_test(bt, n_sims=N_SIMULATIONS)
luck_chart(sims, perf.loc["CAGR", RULE], perf.loc["CAGR", SPY_LABEL], p_value)
plt.show()
print(f"Rule CAGR {perf.loc['CAGR', RULE]:.2%} · random median {np.median(sims):.2%} · "
      f"95th percentile {np.percentile(sims, 95):.2%} · p-value {p_value:.3f}")

## 8. Signal for the upcoming quarter

The rule applied today. The quarter in progress stands in for "the quarter just ended", measured to the latest close,
so the list can change until the quarter actually ends.

In [ ]:
signal = current_signal(prices, cal, SECTORS, LOOKBACK_YEARS, MIN_SEASONAL_AVG, DIP_MARGIN)
buys = signal.index[signal["qualifies"]].tolist()
display(Markdown(f"**Current quarter:** {signal.attrs['current']} · **next quarter:** {signal.attrs['next']} · "
                 f"**seasonal average from** {signal.attrs['years']}  \n"
                 f"**Would buy:** {', '.join(buys) if buys else f'nothing, hold {EMPTY_QUARTER}'}"))
bar_chart(signal["next_quarter_average"], f"Average {signal.attrs['next'][-2:]} return, last {LOOKBACK_YEARS} years",
          f"Seasonal average for {signal.attrs['next']} by sector, {signal.attrs['years']}")
plt.show()
fmt(signal.assign(qualifies=signal["qualifies"].map({True: "✔ buy", False: ""})),
    current_quarter="{:+.1%}", next_quarter_average="{:+.1%}", gap="{:+.1%}")

## 9. Scorecard

Each check has a fixed bar. Passing a few is common for any backtest; the ones that matter most are the
**t-stat**, the **out-of-sample** result and the **luck test**, because they guard against a result that only
looks good by chance.

In [ ]:
scorecard(bt, grid, oos, p_value)

## Notes and limitations

* **Costs.** Each trade pays `COST_BPS` on the amount traded. The ETFs' own fees are already in their prices.
  Taxes are not modelled. Quarterly rotation can realise gains every quarter in a taxable account.
* **Execution.** The signal uses the close on the last trading day of the quarter. Trades happen
  `EXECUTION_LAG_DAYS` trading days later, so the backtest never trades on a price it used to decide.
* **Sector history.** Real Estate (XLRE) starts in 2015 and Communication Services (XLC) in 2018. Each joins once it
  has enough history for the lookback.
* **Concentration.** When only one sector qualifies, the whole portfolio goes into it. Check the average number of
  sectors held and the worst quarter in section 2.
* **Small samples.** Each seasonal average rests on 3 to 10 numbers, and the test covers roughly 60 to 100
  quarters depending on the lookback. Section 5 tries 12 settings, which makes it easy to find one that looks
  good by chance. Sections 6 and 7 are the guard against that.
* **Data.** Prices come from Yahoo Finance, adjusted for splits and dividends. Rerun the notebook to update it.